# RAG 데이터 확인용 노트북

벡터스토어에 저장된 청크, 하이브리드 검색 결과, DART XML 파싱 결과를 셀 단위로 확인하기 위한 노트북입니다.
커널은 `RAG_project (.venv)`를 선택하세요.

In [1]:
import pandas as pd
from rag_core import vectorstore, embeddings, llm
from dart_parser import parse_dart_xml
from hybrid_search import hybrid_search, _detect_company_filter
from ingest import _split_chunks

pd.set_option("display.max_colwidth", 120)

## 1. 저장된 청크 전체 개요
문서(source)별로 몇 개의 청크가 들어있는지 확인합니다.

In [2]:
raw = vectorstore.get(include=["documents", "metadatas"])
df = pd.DataFrame({
    "id": raw["ids"],
    "text": raw["documents"],
    **{k: [m.get(k) for m in raw["metadatas"]] for k in ["source", "chunk_index", "section", "company", "doc_type", "fiscal_period", "ceo_name"]},
})
print("총 청크 수:", len(df))
df.groupby("source").size().sort_values(ascending=False)

총 청크 수: 21242


source
20240325000879.xml    685
20260318001585.xml    673
20240314001683.xml    667
20240314001578.xml    667
20250319000739.xml    641
                     ... 
20220404000994.xml     50
20260330000843.xml     50
20250324000737.xml     50
20240329001212.xml     49
20210331001087.xml     35
Length: 82, dtype: int64

## 2. 특정 문서의 청크 살펴보기
`source_filter`를 원하는 파일명으로 바꿔서 실행하세요. `text`에 회사명/대표이사/사업연도/섹션이 담긴 contextual header가 맨 앞줄에 붙어있는 걸 확인할 수 있습니다.

In [3]:
source_filter = df["source"].iloc[0] if len(df) else None
df[df["source"] == source_filter][["chunk_index", "section", "ceo_name", "text"]]

,chunk_index,section,ceo_name,text
0,0,표지,조덕희,[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | 표지]\n(제 7기 반기)\n\n사업연도 | 2020년 01월 01일 | 부터\n2020년 06월 30일 | 까지\...
1,1,I. 회사의 개요 > 1. 회사의 개요,조덕희,"[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | I. 회사의 개요 > 1. 회사의 개요]\n가. 회사의 법적, 상업적 명칭\n 당사의 명칭은 주식회사 삼양패키징으로..."
2,2,I. 회사의 개요 > 1. 회사의 개요,조덕희,[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | I. 회사의 개요 > 1. 회사의 개요]\n주소: 서울 종로구 종로33길 31 \n전화번호: (02) 740-711...
3,3,I. 회사의 개요 > 1. 회사의 개요,조덕희,[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | I. 회사의 개요 > 1. 회사의 개요]\n당사는 보고서 작성 기준일 현재 당사를 포함하여 19개의 계열회사가 있습...
4,4,I. 회사의 개요 > 1. 회사의 개요,조덕희,[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | I. 회사의 개요 > 1. 회사의 개요]\n사. 신용평가에 관한 사항\n\n평가일 | 평가대상 유가증권 | 신용등급...
...,...,...,...,...
98,98,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,조덕희,[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | XI. 그 밖에 투자자 보호를 위하여 필요한 사항]\n2. \n제\n재\n현\n황\n 등 그 밖의 사항\n가. 작성...
99,99,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,조덕희,[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | XI. 그 밖에 투자자 보호를 위하여 필요한 사항]\n온실가스 배출량(tCO2e) | 에너지 사용량(TJ)\n144...
100,100,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,조덕희,[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | XI. 그 밖에 투자자 보호를 위하여 필요한 사항]\n또한 Aseptic 음료제조시설 투자를 통하여 포장용기의 경량...
101,101,【 전문가의 확인 】 > 1. 전문가의 확인,조덕희,[(주)삼양패키징 | 대표이사 조덕희 | 사업연도 2020.01.01~2020.06.30 | 【 전문가의 확인 】 > 1. 전문가의 확인]\n해당사항 없음


## 3. 하이브리드 검색 테스트
질문을 바꿔가며 어떤 청크가 검색되는지 확인합니다. **꼭 `hybrid_search()`를 써야 회사명 자동 필터링 + BM25 키워드 검색이 적용됩니다.** (`vectorstore.similarity_search`를 직접 쓰면 의미 검색만 되고 이 프로젝트에서 겪었던 문제들이 그대로 재현됩니다.) 질문에 저장된 회사명이 들어있으면 자동으로 그 회사로 범위를 좁혀서 검색하고, `감지된 회사 필터`로 확인할 수 있습니다. `hybrid_search()`를 호출하면 셀 출력 위에 `[검색] ...` 로그가 같이 찍힙니다(순위/점수/섹션).

In [ ]:
question = "삼양패키징 2026년 매출액은 얼마야?"
print("감지된 회사 필터:", _detect_company_filter(question))

docs = hybrid_search(question, k=5)
pd.DataFrame([
    {
        "source": doc.metadata.get("source"),
        "company": doc.metadata.get("company"),
        "section": doc.metadata.get("section"),
        "text": doc.page_content[:200],
    }
    for doc in docs
])

감지된 회사 필터: {'company': {'$in': ['삼양사', '(주)삼양사']}}
[검색] '삼양사 대표이사가 누구야?' | 회사 필터: 삼양사, (주)삼양사
  1위 20221114002256.xml #11 (score 0.0331) | I. 회사의 개요 > 2. 회사의 연혁
  2위 20220816001838.xml #11 (score 0.0331) | I. 회사의 개요 > 2. 회사의 연혁
  3위 20260515001892.xml #10 (score 0.0315) | I. 회사의 개요 > 2. 회사의 연혁
  4위 20260515001892.xml #15 (score 0.0304) | I. 회사의 개요 > 5. 정관에 관한 사항
  5위 20241114000914.xml #338 (score 0.0167) | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항


,source,company,section,text
0,20221114002256.xml,(주)삼양사,I. 회사의 개요 > 2. 회사의 연혁,"[(주)삼양사 | 대표이사 최낙현, 강호성 | 사업연도 2022.01.01~2022.09.30 | I. 회사의 개요 > 2. 회사의 연혁]\n※ | 대표이사 송자량은 2022년 3월 25일 개최된 제11기 정..."
1,20220816001838.xml,(주)삼양사,I. 회사의 개요 > 2. 회사의 연혁,"[(주)삼양사 | 대표이사 최낙현, 강호성 | 사업연도 2022.01.01~2022.06.30 | I. 회사의 개요 > 2. 회사의 연혁]\n※ | 대표이사 송자량은 2022년 3월 25일 개최된 제11기 정..."
2,20260515001892.xml,(주)삼양사,I. 회사의 개요 > 2. 회사의 연혁,[(주)삼양사 | 대표이사 이운익 | 사업연도 2026.01.01~2026.03.31 | I. 회사의 개요 > 2. 회사의 연혁]\n※ | 2026년 3월 26일(목) 개최된 삼양사 제15기 정기주주총회에서 ...
3,20260515001892.xml,(주)삼양사,I. 회사의 개요 > 5. 정관에 관한 사항,[(주)삼양사 | 대표이사 이운익 | 사업연도 2026.01.01~2026.03.31 | I. 회사의 개요 > 5. 정관에 관한 사항]\n가. 정관 변경 이력\n\n정관변경일 | 해당주총명 | 주요변경사항 |...
4,20241114000914.xml,(주)삼양사,VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항,"[(주)삼양사 | 대표이사 최낙현, 강호성 | 사업연도 2024.01.01~2024.09.30 | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항]\n다. 감사위원의 독립성\n\n성 ..."


In [5]:
# 참고용: 벡터(의미) 검색만 썼을 때는 어떻게 다른지 비교 (하이브리드 검색과 결과가 다를 수 있음)
results = vectorstore.similarity_search_with_score(question, k=5)

pd.DataFrame([
    {
        "score": score,
        "source": doc.metadata.get("source"),
        "section": doc.metadata.get("section"),
        "text": doc.page_content[:200],
    }
    for doc, score in results
])

,score,source,section,text
0,0.850194,20241114000914.xml,VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항,"[(주)삼양사 | 대표이사 최낙현, 강호성 | 사업연도 2024.01.01~2024.09.30 | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항]\n다. 감사위원의 독립성\n\n성 ..."
1,0.850724,20230811001664.xml,VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항,"[(주)삼양사 | 대표이사 최낙현, 강호성 | 사업연도 2023.01.01~2023.06.30 | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항]\n나. 감사위원 현황\n\n성명 |..."
2,0.851277,20240814001090.xml,VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항,"[(주)삼양사 | 대표이사 최낙현, 강호성 | 사업연도 2024.01.01~2024.06.30 | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항]\n다. 감사위원의 독립성\n\n성 ..."
3,0.851946,20240514001071.xml,VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항,"[(주)삼양사 | 대표이사 최낙현, 강호성 | 사업연도 2024.01.01~2024.03.31 | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항]\n다. 감사위원의 독립성\n\n성 ..."
4,0.858325,20250319000739.xml,VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항,"[(주)삼양사 | 대표이사 최낙현, 강호성 | 사업연도 2024.01.01~2024.12.31 | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항]\n다. 감사위원의 독립성\n\n성 ..."


## 4. 전체 RAG 파이프라인(query.py의 ask) 실행
검색 + 프롬프트 + GPT 답변까지 한 번에 확인합니다. `query.py`의 `ask()`는 내부적으로 `hybrid_search()`를 사용합니다 (여기서도 `[검색]` 로그가 같이 찍힙니다).

In [6]:
from query import ask

print(ask(question))

[검색] '삼양사 대표이사가 누구야?' | 회사 필터: 삼양사, (주)삼양사
  1위 20221114002256.xml #11 (score 0.0331) | I. 회사의 개요 > 2. 회사의 연혁
  2위 20220816001838.xml #11 (score 0.0331) | I. 회사의 개요 > 2. 회사의 연혁
  3위 20260515001892.xml #10 (score 0.0315) | I. 회사의 개요 > 2. 회사의 연혁
  4위 20260515001892.xml #15 (score 0.0304) | I. 회사의 개요 > 5. 정관에 관한 사항
  5위 20241114000914.xml #338 (score 0.0167) | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항
  6위 20250515002182.xml #342 (score 0.0167) | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항
  7위 20230811001664.xml #300 (score 0.0164) | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항
  8위 20251114002004.xml #363 (score 0.0164) | VI. 이사회 등 회사의 기관에 관한 사항 > 2. 감사제도에 관한 사항
삼양사의 대표이사는 최낙현과 강호성입니다.


## 5. (선택) DART XML 파서 결과만 따로 확인
`ingest.py`를 거치기 전, `dart_parser`가 특정 XML 파일을 어떤 섹션들로 쪼개는지, `ceo_name`/`fiscal_period`를 정확히 뽑는지 미리 확인할 때 사용합니다. 여기서 보이는 `preview`에는 아직 contextual header가 안 붙어있습니다 (그건 `ingest.py`가 청크 분할 이후에 붙이는 것이라서).

In [7]:
from pathlib import Path

xml_path = Path("data/dart_xml/20200814000905.xml")  # 확인하고 싶은 파일로 변경
sections = parse_dart_xml(xml_path)

print("company:", sections[0].metadata["company"], "| ceo_name:", sections[0].metadata["ceo_name"], "| fiscal_period:", sections[0].metadata["fiscal_period"])
pd.DataFrame([
    {"section": d.metadata["section"], "length": len(d.page_content), "preview": d.page_content[:150]}
    for d in sections
])

company: (주)삼양패키징 | ceo_name: 조덕희 | fiscal_period: 2020.01.01~2020.06.30


,section,length,preview
0,표지,320,(제 7기 반기)\n\n사업연도 | 2020년 01월 01일 | 부터\n2020년 06월 30일 | 까지\n\n금융위원회 | \n한국거래소 귀중 | 2020 년 8 월 14 일\n\n제출대상법인 유형 : | ...
1,I. 회사의 개요 > 1. 회사의 개요,973,"가. 회사의 법적, 상업적 명칭\n 당사의 명칭은 주식회사 삼양패키징으로 표기합니다. \n영문으로는 Samyang Packaging Corporation이라고 표기합니다.\n나. 설립일자 및 존속기간\n 당사..."
2,I. 회사의 개요 > 1. 회사의 개요,390,당사는 보고서 작성 기준일 현재 당사를 포함하여 19개의 계열회사가 있습니다.\n\n구분 | 회사명 | 비 고 1 | (주)삼양홀딩스 | 상 장 2 | (주)삼양사 3 | (주)삼양패키징 4 | (주)케이씨아...
3,I. 회사의 개요 > 1. 회사의 개요,238,"사. 신용평가에 관한 사항\n\n평가일 | 평가대상 유가증권 | 신용등급 | 평가회사 | 신용평가등급범위 2018. 7. 27. | 회사채 | A- | 한기\n평㈜, \n나이스신용평가 | AAA~D 2019...."
4,I. 회사의 개요 > 1. 회사의 개요,526,* 신용등급체계 및 부여 의미\n\n신용등급\n체계 | 등급 정의 AAA | 원리금 지급능력이 최상급임 AA | 원리금 지급능력이 매우 우수하지만 AAA의 채권보다는 다소 열위임 A | 원리금 지급능력은 우수...
...,...,...,...
81,X. 이해관계자와의 거래내용,186,"1. \n대주주등과의 영업거래\n\n(단위 : 억원)\n\n거래상대방 \n(회사와의 관계) | 거래내용 | 비 고\n종류 | 기간 | 물품ㆍ서비스명 | 금액\n(주)삼양사\n(지배기업, 최대주주) | 영업거래..."
82,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,1087,1. 주주총회 의사록 요약\n\n주총일자 | 안 건 | 결의내용 2016.03.23 | 1. 제2기 재무제표 승인의 건 | 원안대로 통과 2. 이사 선임의 건 | 원안대로 통과\n기타비상무이사 채완병 3. 감...
83,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,1764,2. \n제\n재\n현\n황\n 등 그 밖의 사항\n가. 작성기준일 이후 발생한 주요 사항\n해\n당사항 없음\n나. 중소기업기준 검토표\n\n당사는 중소기업에 해당하지 않습니다.\n\n다. \n직접금융 자금...
84,【 전문가의 확인 】 > 1. 전문가의 확인,7,해당사항 없음


## 6. 청크 로그 (ingest 전 미리보기, API 호출 없음)
`ingest.py`를 실제로 돌리면(임베딩 비용 발생) 콘솔에 청크별 로그가 찍히는데, 그 내용을 여기서 **임베딩 API 호출 없이** 미리 볼 수 있습니다. `_split_chunks()`가 실제 저장 직전과 동일한 청크 분할(표는 행 단위로, 문단은 글자 수로) 결과를 그대로 돌려줍니다. `is_table`이 `True`인 행이 표에서 나온 청크입니다.

In [8]:
log_path = Path("data/dart_xml/20200814000905.xml")  # 확인하고 싶은 파일로 변경
chunks = _split_chunks(parse_dart_xml(log_path))

print(f"{log_path.name}: {len(chunks)}개 청크 (실제 ingest 시 이 개수로 저장됨)")
pd.DataFrame([
    {
        "chunk_index": i,
        "is_table": c.metadata.get("is_table", False),
        "length": len(c.page_content),
        "section": c.metadata["section"],
        "preview": c.page_content[:80].replace("\n", " "),
    }
    for i, c in enumerate(chunks)
])

20200814000905.xml: 103개 청크 (실제 ingest 시 이 개수로 저장됨)


,chunk_index,is_table,length,section,preview
0,0,False,320,표지,(제 7기 반기) 사업연도 | 2020년 01월 01일 | 부터 2020년 06월 30일 | 까지 금융위원회 | 한국거래소 귀중 | 202
1,1,False,227,I. 회사의 개요 > 1. 회사의 개요,"가. 회사의 법적, 상업적 명칭 당사의 명칭은 주식회사 삼양패키징으로 표기합니다. 영문으로는 Samyang Packaging Corporat"
2,2,False,744,I. 회사의 개요 > 1. 회사의 개요,주소: 서울 종로구 종로33길 31 전화번호: (02) 740-7114 홈페이 지: http://samyangpackaging.co.kr/
3,3,True,390,I. 회사의 개요 > 1. 회사의 개요,당사는 보고서 작성 기준일 현재 당사를 포함하여 19개의 계열회사가 있습니다. 구분 | 회사명 | 비 고 1 | (주)삼양홀딩스 | 상 장 2
4,4,True,238,I. 회사의 개요 > 1. 회사의 개요,사. 신용평가에 관한 사항 평가일 | 평가대상 유가증권 | 신용등급 | 평가회사 | 신용평가등급범위 2018. 7. 27. | 회사채 | A-
...,...,...,...,...,...
98,98,False,781,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,2. 제 재 현 황 등 그 밖의 사항 가. 작성기준일 이후 발생한 주요 사항 해 당사항 없음 나. 중소기업기준 검토표 당사는 중소기업에 해
99,99,False,725,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,"온실가스 배출량(tCO2e) | 에너지 사용량(TJ) 144,902 | 2,957 온실가스 저감활동으로 당사는 사업장의 저효율 에너지설비에 대"
100,100,False,254,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,"또한 Aseptic 음료제조시설 투자를 통하여 포장용기의 경량화를 통한 탄소저감에 기여하고 있으며, 청정연료인 LNG 보일러 전환, LED 조명"
101,101,False,7,【 전문가의 확인 】 > 1. 전문가의 확인,해당사항 없음
